# OCR DateCode — Binary OK/NG Classifier (Colab)

EfficientNet-B0 → 2-class (OK / NG) trained on `char_*/ok` + `char_*/ng` folders.

**Pipeline**
1. Setup & config
2. Upload / extract dataset
3. Dataset class (binary)
4. Augmentation
5. **Visualize data** (distribution + sample grids)
6. Model
7. Train (with class weights for OK/NG imbalance)
8. Evaluation (confusion matrix, AUC, threshold sweep)
9. **NG analysis** (per-char breakdown + FN/FP/Hard NG galleries)
10. ONNX export


## 1. Setup


In [ ]:
# Install dependencies
!pip install -q timm albumentations onnx onnxruntime scikit-learn matplotlib pandas


In [ ]:
# Optional: mount Drive if you store dataset there
# from google.colab import drive
# drive.mount('/content/drive')


## 2. Config — edit paths/hyperparams here


In [ ]:
# === EDIT ME ===
DATASET_ZIP   = '/content/dataset.zip'   # path to your zipped dataset (uploaded to Colab or in Drive)
DATASET_ROOT  = '/content/dataset'       # where to extract → must contain char_*/ folders inside
OUTPUT_DIR    = '/content/runs/ok_ng'

IMAGE_SIZE    = 64
BATCH_SIZE    = 128
EPOCHS        = 30
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
VAL_RATIO     = 0.15
SEED          = 42

import os, random, numpy as np, torch, json
from pathlib import Path
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Upload dataset zip OR skip if you mounted Drive and DATASET_ZIP points there
# from google.colab import files
# uploaded = files.upload()  # select your dataset.zip

# Extract
if not Path(DATASET_ROOT).exists() or not any(Path(DATASET_ROOT).glob('char_*')):
    print('Extracting...')
    !mkdir -p {DATASET_ROOT}
    !unzip -q -o {DATASET_ZIP} -d {DATASET_ROOT}
    # If zip extracts into a nested folder, flatten
    inner = list(Path(DATASET_ROOT).iterdir())
    if len(inner) == 1 and inner[0].is_dir() and not any(Path(DATASET_ROOT).glob('char_*')):
        nested = inner[0]
        for p in nested.iterdir():
            p.rename(Path(DATASET_ROOT) / p.name)
        nested.rmdir()
print('Char folders found:', len(list(Path(DATASET_ROOT).glob('char_*'))))


## 3. Dataset (binary OK/NG)


In [ ]:
import cv2
from collections import defaultdict
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

VALID_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

class CharBinaryDataset(Dataset):
    """Read char_*/ok and char_*/ng. label: 0=OK, 1=NG. Returns (img, label, char_folder, path)."""
    def __init__(self, root, samples=None, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = samples if samples is not None else self._scan(self.root)

    @staticmethod
    def _scan(root):
        out = []
        for char_dir in sorted(Path(root).iterdir()):
            if not char_dir.is_dir() or not char_dir.name.startswith('char_'):
                continue
            for label, sub in [(0, 'ok'), (1, 'ng')]:
                folder = char_dir / sub
                if not folder.is_dir():
                    continue
                for p in sorted(folder.iterdir()):
                    if p.suffix.lower() in VALID_EXT:
                        out.append((str(p), label, char_dir.name))
        return out

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, char_name = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            from PIL import Image
            img = cv2.cvtColor(np.array(Image.open(path).convert('RGB')), cv2.COLOR_RGB2BGR)
        if self.transform is not None:
            img = self.transform(image=img)['image']
        return img, label, char_name, path


def stratified_split(samples, val_ratio=0.15, seed=42):
    """Split per (char, label) so val has both OK and NG of each char."""
    rng = random.Random(seed)
    by_key = defaultdict(list)
    for s in samples:
        by_key[(s[2], s[1])].append(s)
    train, val = [], []
    for k, items in by_key.items():
        items = list(items); rng.shuffle(items)
        n_val = max(1, int(round(len(items) * val_ratio)))
        if n_val >= len(items):
            n_val = max(1, len(items) - 1)
        val.extend(items[:n_val])
        train.extend(items[n_val:])
    return train, val


all_samples = CharBinaryDataset._scan(DATASET_ROOT)
train_samples, val_samples = stratified_split(all_samples, VAL_RATIO, SEED)
print(f'Total: {len(all_samples)}  Train: {len(train_samples)}  Val: {len(val_samples)}')


## 4. Augmentation (matched to synth pipeline)


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Synth pipeline (generate_missing_chars + generate_ng_samples) đã bao:
# rotation ±5°, multi-font, real bg, camera noise, size jitter, blur cho NG, edge noise.
# Augment ở đây chỉ giữ MỘT CHÚT để tránh memorize và phục vụ real-camera subset.

def build_train_tf(size):
    return A.Compose([
        A.Affine(translate_percent=(-0.03, 0.03), rotate=(-2, 2),
                 interpolation=1, cval=255, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.4),
        A.LongestMaxSize(max_size=size, interpolation=1),
        A.PadIfNeeded(min_height=size, min_width=size, border_mode=0, value=(255,255,255)),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])

def build_eval_tf(size):
    return A.Compose([
        A.LongestMaxSize(max_size=size, interpolation=1),
        A.PadIfNeeded(min_height=size, min_width=size, border_mode=0, value=(255,255,255)),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])

train_tf = build_train_tf(IMAGE_SIZE)
val_tf   = build_eval_tf(IMAGE_SIZE)
print('Augment built (minimal). image_size =', IMAGE_SIZE)


## 5. Visualize data

Phân bố OK/NG mỗi char + sample grids


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

per_char = defaultdict(lambda: [0, 0])
for _, l, c in all_samples:
    per_char[c][l] += 1

chars = sorted(per_char.keys())
ok_counts = [per_char[c][0] for c in chars]
ng_counts = [per_char[c][1] for c in chars]

total_ok = sum(ok_counts); total_ng = sum(ng_counts)
print(f'Total OK: {total_ok}  Total NG: {total_ng}  NG ratio: {total_ng/(total_ok+total_ng)*100:.1f}%')
print(f'Chars: {len(chars)}')


In [ ]:
# Stacked bar of samples per char
fig, ax = plt.subplots(figsize=(18, 4))
x = np.arange(len(chars))
ax.bar(x, ok_counts, label='OK', color='steelblue')
ax.bar(x, ng_counts, bottom=ok_counts, label='NG', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(chars, rotation=90, fontsize=7)
ax.set_title('Samples per char (OK + NG stacked)')
ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.show()

# Imbalance ratio per char
fig, ax = plt.subplots(figsize=(18, 3))
ratios = [ng_counts[i] / max(1, ok_counts[i] + ng_counts[i]) for i in range(len(chars))]
ax.bar(x, ratios, color=['tomato' if r > 0.5 else 'steelblue' for r in ratios])
ax.set_xticks(x); ax.set_xticklabels(chars, rotation=90, fontsize=7)
ax.set_ylabel('NG ratio'); ax.set_title('NG ratio per char (>0.5 = more NG than OK)')
ax.axhline(0.5, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout(); plt.show()


In [ ]:
# Sample grid (random OK + NG)
def show_grid(samples_subset, title, n=24, cols=8):
    if not samples_subset:
        print(f'{title}: empty'); return
    rng = np.random.default_rng(0)
    pick = rng.choice(len(samples_subset), size=min(n, len(samples_subset)), replace=False)
    rows_n = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*1.6, rows_n*1.7))
    axes = np.atleast_2d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(pick):
            p, l, c = samples_subset[pick[i]][:3]
            img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f'{c}/{"OK" if l==0 else "NG"}', fontsize=7)
    plt.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

ok_pool = [s for s in all_samples if s[1] == 0]
ng_pool = [s for s in all_samples if s[1] == 1]
show_grid(ok_pool, 'Random OK samples', n=24)
show_grid(ng_pool, 'Random NG samples', n=24)


In [ ]:
# Verify augmentation: show 1 OK + 1 NG with 8 augmented versions each
def show_aug(sample, title, n=8):
    p, l, c = sample[:3]
    img = cv2.imread(p)
    fig, axes = plt.subplots(1, n+1, figsize=((n+1)*1.6, 1.7))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].axis('off'); axes[0].set_title('orig', fontsize=8)
    for i in range(n):
        aug = train_tf(image=img)['image'].numpy().transpose(1,2,0)
        aug = aug * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        aug = np.clip(aug, 0, 1)
        axes[i+1].imshow(aug); axes[i+1].axis('off'); axes[i+1].set_title(f'aug{i+1}', fontsize=8)
    plt.suptitle(f'{title}: {c}/{"OK" if l==0 else "NG"}', fontsize=10)
    plt.tight_layout(); plt.show()

show_aug(ok_pool[0], 'OK augment preview')
show_aug(ng_pool[0], 'NG augment preview')


## 6. Model — EfficientNet-B0 binary head


In [ ]:
import timm, torch.nn as nn

def build_model(num_classes=2):
    m = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
    return m

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: efficientnet_b0  params={n_params:.2f}M')

# Quick forward sanity check
with torch.no_grad():
    out = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE))
print('Forward OK. logits shape:', out.shape)


## 7. Train


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

train_ds = CharBinaryDataset(DATASET_ROOT, samples=train_samples, transform=train_tf)
val_ds   = CharBinaryDataset(DATASET_ROOT, samples=val_samples,   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Class weights for imbalance
n_ok_tr = sum(1 for s in train_samples if s[1] == 0)
n_ng_tr = sum(1 for s in train_samples if s[1] == 1)
w_ok = (n_ok_tr + n_ng_tr) / (2 * max(1, n_ok_tr))
w_ng = (n_ok_tr + n_ng_tr) / (2 * max(1, n_ng_tr))
class_weights = torch.tensor([w_ok, w_ng], dtype=torch.float).to(DEVICE)
print(f'Train OK={n_ok_tr}  NG={n_ng_tr}  weights: OK={w_ok:.3f}  NG={w_ng:.3f}')


In [ ]:
model = build_model().to(DEVICE)
opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

best_score = -1.0
best_path = os.path.join(OUTPUT_DIR, 'best.pt')
history = {'train_loss': [], 'val_loss': [], 'val_ok_pass': [], 'val_ng_catch': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0; n = 0
    for imgs, labels, _, _ in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        loss = loss_fn(logits, labels)
        opt.zero_grad(); loss.backward(); opt.step()
        train_loss += loss.item() * imgs.size(0); n += imgs.size(0)
    train_loss /= max(1, n)
    sched.step()

    model.eval()
    val_loss = 0.0; n = 0; tp = fp = tn = fn = 0
    with torch.no_grad():
        for imgs, labels, _, _ in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            val_loss += loss_fn(logits, labels).item() * imgs.size(0); n += imgs.size(0)
            preds = logits.argmax(1)
            tp += ((preds==1) & (labels==1)).sum().item()
            fn += ((preds==0) & (labels==1)).sum().item()
            tn += ((preds==0) & (labels==0)).sum().item()
            fp += ((preds==1) & (labels==0)).sum().item()
    val_loss /= max(1, n)
    ok_pass = tn / max(1, tn + fp)
    ng_catch = tp / max(1, tp + fn)
    val_acc = (tp + tn) / max(1, tp + tn + fp + fn)
    score = 0.5 * ok_pass + 0.5 * ng_catch
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_ok_pass'].append(ok_pass)
    history['val_ng_catch'].append(ng_catch)
    history['val_acc'].append(val_acc)

    flag = ''
    if score > best_score:
        best_score = score
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'score': score}, best_path)
        flag = ' ← saved best'
    print(f'Ep {epoch+1:02d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'acc={val_acc:.3f}  OK_pass={ok_pass:.3f}  NG_catch={ng_catch:.3f}{flag}')

print(f'\nBest score: {best_score:.4f}  →  {best_path}')


In [ ]:
# Plot history
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
axes[0].plot(history['train_loss'], label='train', color='steelblue')
axes[0].plot(history['val_loss'],   label='val',   color='tomato')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_ok_pass'],  label='OK pass',  color='steelblue')
axes[1].plot(history['val_ng_catch'], label='NG catch', color='tomato')
axes[1].plot(history['val_acc'],      label='accuracy', color='gray', linestyle='--')
axes[1].set_title('Val rates'); axes[1].set_xlabel('epoch'); axes[1].set_ylim(0, 1.02); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 8. Evaluation


In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve

# Load best
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded best (epoch {ckpt["epoch"]+1}, score {ckpt["score"]:.4f})')

# Get all val probabilities
all_probs, all_labels, all_chars, all_paths = [], [], [], []
with torch.no_grad():
    for imgs, labels, chars, paths in val_loader:
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())
        all_chars.extend(list(chars))
        all_paths.extend(list(paths))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
print(f'Val: {len(all_probs)} samples')
auc = roc_auc_score(all_labels, all_probs)
print(f'ROC-AUC: {auc:.4f}')


In [ ]:
# Threshold sweep — pick best by balanced score (0.5*OK_pass + 0.5*NG_catch)
ths = np.linspace(0.05, 0.95, 91)
best_th, best_balanced = 0.5, -1.0
sweep_rows = []
for th in ths:
    pred = (all_probs >= th).astype(int)
    tp = int(((pred==1) & (all_labels==1)).sum())
    fp = int(((pred==1) & (all_labels==0)).sum())
    tn = int(((pred==0) & (all_labels==0)).sum())
    fn = int(((pred==0) & (all_labels==1)).sum())
    okp = tn / max(1, tn + fp)
    ngc = tp / max(1, tp + fn)
    sc = 0.5 * okp + 0.5 * ngc
    sweep_rows.append({'th': th, 'ok_pass': okp, 'ng_catch': ngc, 'balanced': sc})
    if sc > best_balanced:
        best_balanced = sc; best_th = th

sweep_df = pd.DataFrame(sweep_rows)
print(f'Best threshold: {best_th:.3f}  →  OK_pass={sweep_df.loc[sweep_df.th==best_th,"ok_pass"].values[0]:.4f}  '
      f'NG_catch={sweep_df.loc[sweep_df.th==best_th,"ng_catch"].values[0]:.4f}  '
      f'balanced={best_balanced:.4f}')

# Threshold sweep plot
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sweep_df.th, sweep_df.ok_pass,  label='OK pass',  color='steelblue')
ax.plot(sweep_df.th, sweep_df.ng_catch, label='NG catch', color='tomato')
ax.plot(sweep_df.th, sweep_df.balanced, label='balanced', color='black', linestyle='--')
ax.axvline(best_th, color='green', linestyle=':', label=f'best={best_th:.2f}')
ax.set_xlabel('threshold (P_NG)'); ax.set_ylabel('rate'); ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Threshold sweep')
plt.tight_layout(); plt.show()


In [ ]:
# Final report at best threshold
preds = (all_probs >= best_th).astype(int)
print(classification_report(all_labels, preds, target_names=['OK','NG'], digits=4))

cm = confusion_matrix(all_labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Confusion matrix
ax = axes[0]
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred OK', 'Pred NG'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['True OK', 'True NG'])
ax.set_title(f'Confusion Matrix (th={best_th:.2f})')

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
ax = axes[1]
ax.plot(fpr, tpr, color='tomato', label=f'AUC={auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 9. NG analysis

Phân tích sâu các sample NG:
- **Per-char breakdown** — char nào model bắt NG kém nhất?
- **False Negatives** — NG bị model bỏ sót (predict OK) — **nguy hiểm nhất**
- **False Positives** — OK bị model reject nhầm
- **Hard NG** — NG đúng nhưng confidence thấp (gần threshold)
- **Easy NG** — NG đúng với confidence cao


In [ ]:
# Build analysis dataframe
df = pd.DataFrame({
    'path': all_paths,
    'char': all_chars,
    'label': all_labels,
    'prob_ng': all_probs,
})
df['pred']    = (df['prob_ng'] >= best_th).astype(int)
df['correct'] = df['pred'] == df['label']
df['kind']    = df.apply(lambda r: ('TN','FN','FP','TP')[r['label']*2 + r['pred']], axis=1)
print(df['kind'].value_counts().to_string())


In [ ]:
# Per-char breakdown
rows = []
for ch, g in df.groupby('char'):
    ok = g[g['label']==0]; ng = g[g['label']==1]
    rows.append({
        'char': ch,
        'n_ok': len(ok),
        'n_ng': len(ng),
        'ok_pass':  (ok['pred']==0).mean()  if len(ok) > 0 else float('nan'),
        'ng_catch': (ng['pred']==1).mean()  if len(ng) > 0 else float('nan'),
        'fn_count': int(((ng['pred']==0)).sum()) if len(ng) > 0 else 0,
        'fp_count': int(((ok['pred']==1)).sum()) if len(ok) > 0 else 0,
    })
per_char_df = pd.DataFrame(rows)

print('=== 15 chars worst NG catch (NG bị bỏ sót nhiều nhất) ===')
print(per_char_df.dropna(subset=['ng_catch']).sort_values('ng_catch').head(15).to_string(index=False))
print()
print('=== 15 chars worst OK pass (OK bị reject nhầm nhiều nhất) ===')
print(per_char_df.dropna(subset=['ok_pass']).sort_values('ok_pass').head(15).to_string(index=False))


In [ ]:
# Per-char OK pass / NG catch bar chart
plot_df = per_char_df.sort_values('char')
fig, ax = plt.subplots(figsize=(18, 4))
x = np.arange(len(plot_df))
w = 0.4
ax.bar(x - w/2, plot_df.ok_pass.fillna(0),  width=w, label='OK pass',  color='steelblue')
ax.bar(x + w/2, plot_df.ng_catch.fillna(0), width=w, label='NG catch', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(plot_df.char, rotation=90, fontsize=7)
ax.set_ylim(0, 1.02); ax.set_ylabel('rate'); ax.legend()
ax.set_title('Per-char OK pass vs NG catch (val set)')
ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout(); plt.show()


In [ ]:
# Helper: gallery showing image + char + P_NG
def gallery(sub_df, title, n=24, cols=8, sort_by=None, ascending=True):
    sub = sub_df.copy()
    if sort_by is not None:
        sub = sub.sort_values(sort_by, ascending=ascending)
    sub = sub.head(n)
    if len(sub) == 0:
        print(f'{title}: empty'); return
    rows_n = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*1.7, rows_n*1.9))
    axes = np.atleast_2d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(sub):
            r = sub.iloc[i]
            img = cv2.cvtColor(cv2.imread(r['path']), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            tag = f"{r['char']}\nP_NG={r['prob_ng']:.2f}"
            color = 'red' if r['kind'] in ('FN','FP') else 'black'
            ax.set_title(tag, fontsize=8, color=color)
    plt.suptitle(f'{title}  (n={len(sub)})', fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# False Negatives — NG escaped as OK (most dangerous)
fn_df = df[df['kind'] == 'FN']
print(f'False Negatives: {len(fn_df)} / {(df.label==1).sum()} NG samples')
gallery(fn_df, 'False Negatives — NG escaped as OK', n=24, sort_by='prob_ng', ascending=False)


In [ ]:
# False Positives — OK rejected as NG
fp_df = df[df['kind'] == 'FP']
print(f'False Positives: {len(fp_df)} / {(df.label==0).sum()} OK samples')
gallery(fp_df, 'False Positives — OK rejected as NG', n=24, sort_by='prob_ng', ascending=True)


In [ ]:
# Hardest NG (correctly predicted but low margin)
tp_df = df[df['kind'] == 'TP'].copy()
tp_df['margin'] = tp_df['prob_ng'] - best_th
gallery(tp_df, 'Hardest correctly-classified NG (low margin)', n=24, sort_by='margin', ascending=True)

# Most confident NG (sanity check)
gallery(tp_df, 'Most confident NG (high P_NG)', n=24, sort_by='prob_ng', ascending=False)


In [ ]:
# Save analysis CSVs for offline review
fn_path = os.path.join(OUTPUT_DIR, 'false_negatives.csv')
fp_path = os.path.join(OUTPUT_DIR, 'false_positives.csv')
per_char_path = os.path.join(OUTPUT_DIR, 'per_char_metrics.csv')

fn_df[['path','char','prob_ng']].to_csv(fn_path, index=False)
fp_df[['path','char','prob_ng']].to_csv(fp_path, index=False)
per_char_df.to_csv(per_char_path, index=False)
print('Saved:')
print(' -', fn_path)
print(' -', fp_path)
print(' -', per_char_path)


## 10. ONNX export + metadata


In [ ]:
# Export ONNX (single-file, no .data sidecar)
import onnx
model.eval()
onnx_path = os.path.join(OUTPUT_DIR, 'ok_ng_model.onnx')
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)

# Force single-file: load (resolving any external data) then re-save without external data
m = onnx.load(onnx_path, load_external_data=True)
onnx.save_model(m, onnx_path, save_as_external_data=False)

# Cleanup any leftover .data sidecar files
for sidecar in Path(OUTPUT_DIR).glob('ok_ng_model*.data'):
    sidecar.unlink()
for sidecar in Path(OUTPUT_DIR).glob('ok_ng_model.onnx.data'):
    sidecar.unlink()

size_mb = os.path.getsize(onnx_path) / 1e6
print(f'Exported {onnx_path}  ({size_mb:.2f} MB, single-file)')

# Verify ONNX
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
test_in = np.random.randn(32, 3, IMAGE_SIZE, IMAGE_SIZE).astype(np.float32)
out = sess.run(None, {'input': test_in})[0]
print('ONNX output shape:', out.shape, ' (sanity check OK)')


In [ ]:
# Metadata for inference (preprocessing + threshold)
meta_path = os.path.join(OUTPUT_DIR, 'model_meta.json')
meta = {
    'image_size': IMAGE_SIZE,
    'threshold': float(best_th),
    'normalization': {
        'mean': list(IMAGENET_MEAN),
        'std':  list(IMAGENET_STD),
    },
    'classes': {'0': 'OK', '1': 'NG'},
    'metrics': {
        'auc': float(auc),
        'balanced_score': float(best_balanced),
    },
    'training': {
        'backbone': 'efficientnet_b0',
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
    },
}
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved', meta_path)
print(json.dumps(meta, indent=2))


In [ ]:
# Download artifacts
from google.colab import files
files.download(onnx_path)
files.download(meta_path)
files.download(per_char_path)
files.download(fn_path)
files.download(fp_path)


## Inference snippet (paste into desktop app or any Python project)

```python
import cv2, numpy as np, json, onnxruntime as ort

sess = ort.InferenceSession('ok_ng_model.onnx', providers=['CPUExecutionProvider'])
meta = json.load(open('model_meta.json'))
SIZE = meta['image_size']; TH = meta['threshold']
MEAN = np.array(meta['normalization']['mean'], dtype=np.float32)
STD  = np.array(meta['normalization']['std'],  dtype=np.float32)

def preprocess(bgr):
    h, w = bgr.shape[:2]
    s = SIZE / max(h, w)
    nh, nw = int(round(h*s)), int(round(w*s))
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((SIZE, SIZE, 3), 255, dtype=np.uint8)
    y0, x0 = (SIZE-nh)//2, (SIZE-nw)//2
    canvas[y0:y0+nh, x0:x0+nw] = img
    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - MEAN) / STD
    return rgb.transpose(2, 0, 1)[None]   # (1,3,SIZE,SIZE)

def predict(bgr):
    x = preprocess(bgr).astype(np.float32)
    logits = sess.run(None, {'input': x})[0][0]
    e = np.exp(logits - logits.max()); p = e / e.sum()
    p_ng = float(p[1])
    return ('NG' if p_ng >= TH else 'OK'), p_ng
```
